# Module 5: KV Cache 管理

## 学习目标
- 理解 Naive Cache Manager 的工作原理
- 掌握 Radix Cache 的核心概念
- 学习前缀匹配和缓存复用
- 理解缓存驱逐策略

---

## 5.1 代码位置

```
mini-sglang/python/minisgl/kvcache/
├── base.py            # 基类定义
├── naive_manager.py   # 简单缓存管理器
├── radix_manager.py   # Radix Tree 缓存管理器
└── mha_pool.py        # KV Cache 存储池
```

## 5.2 为什么需要智能缓存管理?

在 LLM 服务中，常见的场景是多个请求共享相同的前缀：

```
请求 1: "You are a helpful assistant. What is AI?"
请求 2: "You are a helpful assistant. How does ML work?"
请求 3: "You are a helpful assistant. Explain deep learning."
          ↑─────────────────────────↑
          相同的 system prompt 前缀
```

**问题**: 如果每个请求都重新计算 KV Cache，会有大量重复计算。

**解决方案**: Radix Cache - 缓存并复用相同前缀的 KV Cache。

In [ ]:
import torch
from typing import Dict, List, Tuple
from dataclasses import dataclass
from abc import ABC, abstractmethod

@dataclass
class SizeInfo:
    """缓存大小信息"""
    evictable_size: int  # 可驱逐的大小
    protected_size: int  # 受保护的大小 (正在使用)

@dataclass
class BaseCacheHandle:
    """缓存句柄，表示一个缓存匹配结果"""
    cached_len: int  # 匹配到的缓存长度

class BaseCacheManager(ABC):
    """缓存管理器基类"""
    
    @abstractmethod
    def match_prefix(self, input_ids: torch.Tensor) -> Tuple[BaseCacheHandle, torch.Tensor]:
        """匹配最长前缀，返回 (句柄, 缓存的页索引)"""
        ...
    
    @abstractmethod
    def lock_handle(self, handle: BaseCacheHandle, unlock: bool = False) -> None:
        """锁定/解锁缓存句柄"""
        ...
    
    @abstractmethod
    def insert_prefix(self, input_ids: torch.Tensor, indices: torch.Tensor) -> int:
        """插入新的前缀到缓存"""
        ...
    
    @abstractmethod
    def evict(self, size: int) -> torch.Tensor:
        """驱逐指定大小的缓存，返回被驱逐的页索引"""
        ...

print("BaseCacheManager 定义了 4 个核心方法:")
print("  1. match_prefix(): 查找最长匹配前缀")
print("  2. lock_handle(): 保护正在使用的缓存")
print("  3. insert_prefix(): 插入新的缓存条目")
print("  4. evict(): 驱逐过期缓存释放空间")

## 5.3 Naive Cache Manager

最简单的缓存管理器：不进行前缀复用，每个请求独立分配。

In [ ]:
@dataclass
class NaiveCacheHandle(BaseCacheHandle):
    """Naive 缓存句柄 - 总是返回空匹配"""
    pass

class NaiveCacheManager(BaseCacheManager):
    """简单缓存管理器 - 不进行前缀复用"""
    
    def __init__(self, device: torch.device, num_pages: int):
        self.device = device
        self.num_pages = num_pages
        self.free_pages = list(range(num_pages))
        self.used_pages = 0
    
    def match_prefix(self, input_ids: torch.Tensor) -> Tuple[NaiveCacheHandle, torch.Tensor]:
        """Naive 实现：总是返回空匹配"""
        return NaiveCacheHandle(cached_len=0), torch.empty(0, dtype=torch.int32)
    
    def lock_handle(self, handle: BaseCacheHandle, unlock: bool = False) -> None:
        """Naive 实现：不需要锁定"""
        pass
    
    def insert_prefix(self, input_ids: torch.Tensor, indices: torch.Tensor) -> int:
        """Naive 实现：不缓存"""
        return 0
    
    def evict(self, size: int) -> torch.Tensor:
        """Naive 实现：直接返回空闲页"""
        if size > len(self.free_pages):
            raise RuntimeError(f"Not enough free pages: need {size}, have {len(self.free_pages)}")
        evicted = self.free_pages[:size]
        self.free_pages = self.free_pages[size:]
        return torch.tensor(evicted, dtype=torch.int32)
    
    def allocate(self, size: int) -> torch.Tensor:
        """分配页"""
        if size > len(self.free_pages):
            raise RuntimeError(f"Not enough free pages")
        allocated = self.free_pages[:size]
        self.free_pages = self.free_pages[size:]
        self.used_pages += size
        return torch.tensor(allocated, dtype=torch.int32)
    
    def free(self, pages: torch.Tensor) -> None:
        """释放页"""
        self.free_pages.extend(pages.tolist())
        self.used_pages -= len(pages)

# 测试
naive_manager = NaiveCacheManager(torch.device('cpu'), num_pages=100)
input_ids = torch.tensor([1, 2, 3, 4, 5], dtype=torch.int32)

handle, cached_indices = naive_manager.match_prefix(input_ids)
print(f"Naive Cache 匹配结果:")
print(f"  cached_len: {handle.cached_len}")
print(f"  cached_indices: {cached_indices.tolist()}")
print(f"\n每个请求都需要从头计算，没有前缀复用")

## 5.4 Radix Tree 数据结构

Radix Tree (压缩前缀树) 是一种高效存储字符串前缀的数据结构。

```
普通 Trie (前缀树):          Radix Tree (压缩前缀树):
        root                       root
       /    \                     /    \
      t      b                  test   bear
      |      |                  /  \      \
      e      e                 er   s     s
      |      |                              
      s      a                              
     / \     |
    t   s    r
    |        |
    er       s

存储: test, tester, tests, bear, bears
```

在 Mini-SGLang 中，每个节点存储:
- `key`: token IDs 序列
- `value`: 对应的 KV Cache 页索引
- `ref_count`: 引用计数 (保护正在使用的节点)
- `timestamp`: 最后访问时间 (用于 LRU 驱逐)

In [ ]:
import time

class RadixTreeNode:
    """Radix Tree 节点"""
    counter = 0
    
    def __init__(self):
        self.children: Dict[int, 'RadixTreeNode'] = {}  # token_id -> child_node
        self._parent: 'RadixTreeNode' = None
        self.ref_count: int = 0  # 引用计数
        self.uuid = RadixTreeNode.counter
        RadixTreeNode.counter += 1
        self.timestamp = time.monotonic_ns()  # 最后访问时间
        
        # key 和 value
        self._key: torch.Tensor = None  # token IDs
        self._value: torch.Tensor = None  # KV cache 页索引
    
    def set_key_value(self, key: torch.Tensor, value: torch.Tensor):
        assert len(key) == len(value)
        self._key = key
        self._value = value
    
    def set_parent(self, parent: 'RadixTreeNode'):
        self._parent = parent
        # 将自己添加到父节点的 children 中
        parent.children[int(self._key[0].item())] = self
    
    @property
    def length(self) -> int:
        return len(self._key) if self._key is not None else 0
    
    def is_root(self) -> bool:
        return self._parent is None
    
    def is_leaf(self) -> bool:
        return len(self.children) == 0
    
    def get_match_len(self, input_ids: torch.Tensor) -> int:
        """计算与输入匹配的长度"""
        min_len = min(len(self._key), len(input_ids))
        for i in range(min_len):
            if self._key[i] != input_ids[i]:
                return i
        return min_len
    
    def __repr__(self):
        return f"Node(uuid={self.uuid}, key={self._key.tolist() if self._key is not None else []}, ref={self.ref_count})"

# 可视化 Radix Tree
def visualize_radix_tree(root: RadixTreeNode, prefix: str = ""):
    """可视化 Radix Tree 结构"""
    if root.is_root():
        print(f"{prefix}[ROOT] (ref={root.ref_count})")
    else:
        key_str = root._key.tolist() if root._key is not None else []
        print(f"{prefix}├─ {key_str} (ref={root.ref_count})")
    
    children = list(root.children.values())
    for i, child in enumerate(children):
        is_last = (i == len(children) - 1)
        new_prefix = prefix + ("   " if is_last else "│  ")
        visualize_radix_tree(child, new_prefix)

In [ ]:
@dataclass
class RadixCacheHandle(BaseCacheHandle):
    """Radix Cache 句柄"""
    node: RadixTreeNode

class RadixCacheManager(BaseCacheManager):
    """基于 Radix Tree 的缓存管理器"""
    
    def __init__(self, device: torch.device):
        self.device = device
        self.root_node = RadixTreeNode()
        self.root_node.ref_count = 1  # root 永远受保护
        self.evictable_size = 0
        self.protected_size = 0
    
    def _walk(self, input_ids: torch.Tensor) -> Tuple[RadixTreeNode, int]:
        """遍历树，返回 (最后匹配的节点, 匹配长度)"""
        prefix_len = 0
        node = self.root_node
        
        while prefix_len < len(input_ids):
            this_id = int(input_ids[prefix_len].item())
            
            # 找不到对应的子节点
            if this_id not in node.children:
                return node, prefix_len
            
            node = node.children[this_id]
            match_len = node.get_match_len(input_ids[prefix_len:])
            prefix_len += match_len
            
            # 部分匹配 - 需要分裂节点
            if match_len != node.length:
                # 这里简化处理，实际需要分裂
                return node, prefix_len
            
            # 更新时间戳
            node.timestamp = time.monotonic_ns()
        
        return node, prefix_len
    
    def match_prefix(self, input_ids: torch.Tensor) -> Tuple[RadixCacheHandle, torch.Tensor]:
        """匹配最长前缀"""
        node, prefix_len = self._walk(input_ids)
        
        if prefix_len == 0:
            return RadixCacheHandle(0, self.root_node), torch.empty(0, dtype=torch.int32)
        
        # 收集所有匹配节点的 value (KV cache 页索引)
        values = []
        current = node
        while not current.is_root():
            values.append(current._value)
            current = current._parent
        values.reverse()
        
        return RadixCacheHandle(prefix_len, node), torch.cat(values)
    
    def insert_prefix(self, input_ids: torch.Tensor, indices: torch.Tensor) -> int:
        """插入新的前缀"""
        node, prefix_len = self._walk(input_ids)
        
        if prefix_len < len(input_ids):
            # 创建新节点
            new_node = RadixTreeNode()
            new_node.set_key_value(input_ids[prefix_len:], indices[prefix_len:])
            new_node.set_parent(node)
            self.evictable_size += new_node.length
        
        return prefix_len
    
    def lock_handle(self, handle: BaseCacheHandle, unlock: bool = False):
        """锁定/解锁缓存"""
        assert isinstance(handle, RadixCacheHandle)
        node = handle.node
        
        while not node.is_root():
            node = node._parent
            if unlock:
                node.ref_count -= 1
            else:
                node.ref_count += 1
    
    def evict(self, size: int) -> torch.Tensor:
        """驱逐缓存 (LRU 策略)"""
        # 简化实现 - 实际需要使用堆来实现 LRU
        return torch.empty(0, dtype=torch.int32)

In [ ]:
# 演示 Radix Cache 的工作过程
radix_manager = RadixCacheManager(torch.device('cpu'))

# 模拟 3 个有相同前缀的请求
system_prompt = [1, 2, 3, 4, 5]  # "You are a helpful assistant."
req1_ids = torch.tensor(system_prompt + [10, 11, 12], dtype=torch.int32)  # + "What is AI?"
req2_ids = torch.tensor(system_prompt + [20, 21], dtype=torch.int32)      # + "How?"
req3_ids = torch.tensor(system_prompt + [10, 11, 30], dtype=torch.int32)  # + "What is ML?"

print("=== 请求 1: 首次请求 ===")
handle1, cached1 = radix_manager.match_prefix(req1_ids)
print(f"匹配长度: {handle1.cached_len} (应该是 0，因为缓存为空)")

# 模拟完成 prefill，插入缓存
pages1 = torch.arange(len(req1_ids), dtype=torch.int32) * 10  # 模拟页索引
radix_manager.insert_prefix(req1_ids, pages1)
print(f"插入缓存: {req1_ids.tolist()} -> pages {pages1.tolist()}")

print("\n=== 请求 2: 有相同 system prompt ===")
handle2, cached2 = radix_manager.match_prefix(req2_ids)
print(f"匹配长度: {handle2.cached_len} tokens")
print(f"复用的 KV cache 页: {cached2.tolist()}")
print(f"节省了 {handle2.cached_len} 个 token 的计算!")

print("\n=== 请求 3: 前缀更长 ===")
handle3, cached3 = radix_manager.match_prefix(req3_ids)
print(f"匹配长度: {handle3.cached_len} tokens")
print(f"复用的 KV cache 页: {cached3.tolist()}")

## 5.5 缓存的生命周期

```
┌──────────────────────────────────────────────────────────────────────────┐
│                           缓存生命周期                                   │
├──────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│  1. 请求到达                                                             │
│     ↓                                                                    │
│  2. match_prefix() - 查找最长匹配前缀                                    │
│     ↓                                                                    │
│  3. lock_handle() - 保护匹配的缓存节点 (ref_count++)                     │
│     ↓                                                                    │
│  4. 执行 prefill (只计算未缓存的部分)                                    │
│     ↓                                                                    │
│  5. 执行 decode 循环                                                     │
│     ↓                                                                    │
│  6. 请求完成                                                             │
│     ↓                                                                    │
│  7. insert_prefix() - 将完整序列插入缓存                                 │
│     ↓                                                                    │
│  8. unlock_handle() - 解除保护 (ref_count--)                             │
│     ↓                                                                    │
│  9. 缓存可被后续请求复用或被 evict()                                     │
│                                                                          │
└──────────────────────────────────────────────────────────────────────────┘
```

## 5.6 LRU 驱逐策略

当内存不足时，需要驱逐一些缓存来释放空间。

### 驱逐规则:
1. 只驱逐 `ref_count == 0` 的节点 (没有请求在使用)
2. 只驱逐叶子节点 (防止破坏树结构)
3. 优先驱逐 `timestamp` 最早的节点 (LRU)

In [ ]:
import heapq

def lru_eviction_demo():
    """演示 LRU 驱逐策略"""
    # 创建一些模拟的缓存节点
    nodes = []
    for i in range(5):
        node = RadixTreeNode()
        node.set_key_value(
            torch.tensor([i * 10 + j for j in range(3)], dtype=torch.int32),
            torch.tensor([i * 100 + j for j in range(3)], dtype=torch.int32)
        )
        time.sleep(0.001)  # 确保时间戳不同
        nodes.append(node)
    
    # 设置一些节点被保护
    nodes[1].ref_count = 1  # 正在使用
    nodes[3].ref_count = 2  # 多个请求在使用
    
    print("缓存节点状态:")
    for i, node in enumerate(nodes):
        status = "受保护" if node.ref_count > 0 else "可驱逐"
        print(f"  Node {i}: ref_count={node.ref_count}, status={status}")
    
    # 收集可驱逐的节点
    evictable = [(node.timestamp, node) for node in nodes if node.ref_count == 0]
    heapq.heapify(evictable)
    
    print(f"\n可驱逐节点数: {len(evictable)}")
    print("\nLRU 驱逐顺序 (按时间戳):")
    while evictable:
        ts, node = heapq.heappop(evictable)
        print(f"  驱逐 Node {node.uuid}, 释放 {node.length} 个页")

lru_eviction_demo()

## 5.7 Radix Cache vs Naive Cache 对比

| 特性 | Naive Cache | Radix Cache |
|------|-------------|-------------|
| 前缀复用 | 不支持 | 支持 |
| 内存效率 | 低 | 高 |
| 实现复杂度 | 简单 | 复杂 |
| 适用场景 | 测试/Benchmark | 生产环境 |
| 查找复杂度 | O(1) | O(prefix_len) |

In [ ]:
# 对比演示
def compare_cache_managers():
    """对比两种缓存管理器"""
    system_prompt = torch.tensor([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=torch.int32)
    
    # 10 个请求，都有相同的 system prompt
    requests = [
        torch.cat([system_prompt, torch.tensor([100 + i, 101 + i], dtype=torch.int32)])
        for i in range(10)
    ]
    
    print("场景: 10 个请求，共享 10 token 的 system prompt")
    print("="*60)
    
    # Naive Cache
    naive_manager = NaiveCacheManager(torch.device('cpu'), num_pages=200)
    naive_total_compute = 0
    for req in requests:
        handle, _ = naive_manager.match_prefix(req)
        naive_total_compute += len(req) - handle.cached_len
    
    print(f"\nNaive Cache:")
    print(f"  总计算量: {naive_total_compute} tokens")
    print(f"  每个请求都要计算全部 {len(requests[0])} tokens")
    
    # Radix Cache
    radix_manager = RadixCacheManager(torch.device('cpu'))
    radix_total_compute = 0
    for i, req in enumerate(requests):
        handle, cached_indices = radix_manager.match_prefix(req)
        compute_needed = len(req) - handle.cached_len
        radix_total_compute += compute_needed
        
        # 模拟 prefill 完成后插入缓存
        if i == 0:  # 只有第一个请求需要插入
            pages = torch.arange(len(req), dtype=torch.int32)
            radix_manager.insert_prefix(req, pages)
    
    print(f"\nRadix Cache:")
    print(f"  总计算量: {radix_total_compute} tokens")
    print(f"  第一个请求: 计算 {len(requests[0])} tokens")
    print(f"  后续请求: 只计算 {len(requests[0]) - len(system_prompt)} tokens")
    
    savings = (naive_total_compute - radix_total_compute) / naive_total_compute * 100
    print(f"\n节省计算量: {savings:.1f}%")

compare_cache_managers()

## 5.8 小结

### 核心要点:

1. **Naive Cache**:
   - 简单的分配/释放管理
   - 不支持前缀复用
   - 适合测试和 benchmark

2. **Radix Cache**:
   - 使用 Radix Tree 存储缓存
   - 支持高效的前缀匹配
   - 自动复用相同前缀的 KV Cache
   - 使用 LRU 策略进行驱逐

3. **缓存保护机制**:
   - 使用 `ref_count` 保护正在使用的缓存
   - 只有 `ref_count == 0` 的节点才能被驱逐

4. **性能优势**:
   - 共享 system prompt 可节省大量计算
   - 对于多轮对话尤其有效

---

**下一步**: [Module 6: Scheduler 调度逻辑](./06_scheduler_logic.ipynb) - 学习请求调度和批处理策略。